In [0]:
from pyspark.sql.functions import col

fact_df = spark.table("medical_project.gold.fact_encounters")
date_df = spark.table("medical_project.gold.dim_date")

In [0]:
from pyspark.sql.functions import to_date

df = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

from pyspark.sql.functions import quarter

df = df.withColumn(
    "quarter",
    quarter(col("encounter_date"))
)

df = df.withColumn("year", col("year").cast("int")) \
       .withColumn("month", col("month").cast("int"))

In [0]:
# Save 
from pyspark.sql.functions import count, sum, col, coalesce, lit

# Recreate cube without problematic fillna
cube_for_save = df.cube(
    "year",
    "month",
    "payer_id",
    "encounter_class"
).agg(
    count("encounter_id").alias("encounter_count"),
    sum("total_cost").alias("total_cost")
)

# Handle nulls with proper type conversions
cube_for_save = cube_for_save \
    .withColumn("year", coalesce(col("year").cast("string"), lit("ALL"))) \
    .withColumn("month", coalesce(col("month").cast("string"), lit("ALL"))) \
    .withColumn("payer_id", coalesce(col("payer_id"), lit("ALL"))) \
    .withColumn("encounter_class", coalesce(col("encounter_class"), lit("ALL")))

cube_for_save.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.encounter_cube")